## Conversion from Keras HGQ2 model to HLS4ML model (Verilog)

In [1]:
import os, subprocess
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Keras 3
import keras

# Package imports
from bedposip.data import load_dataset
from bedposip.utils import inspect_hls4ml_precision, class_report_metric

import hls4ml
from hls4ml.model.profiling import numerical, compare



PROFILE = False
BUILD_IP = True
RESET_BUILD = True

I0000 00:00:1786050226.130698  114995 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Configure matplotlib for PDF export with serif font
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern Serif",
})

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)
keras.utils.set_random_seed(42)

# Constant number of postures
NUM_CLASSES = 3

output_dir = str(Path("output/export_ip").resolve())

X_TEST_PATH = str(Path("input/X_test.npy").resolve())
Y_TEST_PATH = str(Path("input/y_test.npy").resolve())

## Data

### Loading

In [3]:
# Load datasets using package function
X_train, y_train, X_val, y_val, X_test, y_test = load_dataset()


Dataset shapes:
  Train: (119999, 8, 8, 1), labels: (119999,)
  Val:   (60000, 8, 8, 1), labels: (60000,)
  Test:  (120001, 8, 8, 1), labels: (120001,)


### Encoding

In [4]:
class_names = ["lateral_L", "lateral_R", "supine"]

## Model Loading

In [5]:
model_path = "output/posture_classifier_qat.keras"
model_qat = keras.models.load_model(model_path)
model_qat.summary()

I0000 00:00:1786050228.699777  114995 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4994 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:10:00.0, compute capability: 8.6


Model: "posture_classifier_qat"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pressure_map (InputLayer)       │ (None, 8, 8, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_1 (QConv2D)               │ (None, 6, 6, 20)       │           726 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qbn_conv_1                      │ (None, 6, 6, 20)       │           206 │
│ (QBatchNormalization)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_act_1 (Activation)        │ (None, 6, 6, 20)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_2 (QConv2D)               │ (None, 4, 4, 20)       │        14,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qbn_conv_2                      │ (None, 4, 4, 20)       │           206 │
│ (QBatchNormalization)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_act_2 (Activation)        │ (None, 4, 4, 20)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_3 (QConv2D)               │ (None, 2, 2, 28)       │        20,166 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qbn_conv_3                      │ (None, 2, 2, 28)       │           286 │
│ (QBatchNormalization)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qconv_act_3 (Activation)        │ (None, 2, 2, 28)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_max_pooling2d_2               │ (None, 1, 1, 28)       │             6 │
│ (QMaxPooling2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 28)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qdense_1 (QDense)               │ (None, 28)             │         3,142 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qbn_dense_1                     │ (None, 28)             │           286 │
│ (QBatchNormalization)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 28)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qdense_2 (QDense)               │ (None, 24)             │         2,694 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ qbn_dense_2                     │ (None, 24)             │           246 │
│ (QBatchNormalization)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (QDense)                 │ (None, 3)              │           306 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,248 (383.97 KB)

 Trainable params: 31,785 (124.16 KB)

 Non-trainable params: 10,891 (11.48 KB)

 Optimizer params: 63,572 (248.33 KB)

## Conversion

In [6]:
# Make Vitis available in path
script = "/home/jeison/AMD/Vitis/2023.2/settings64.sh"

for line in subprocess.check_output(f'source "{script}" >/dev/null && env', shell=True, executable='/bin/bash', text=True).splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        os.environ[k] = v

## HLS4ML

In [7]:
config = hls4ml.utils.config_from_keras_model(model_qat, granularity='name', backend='Vitis')

config['Model']['Strategy'] = 'Resource'
config['Model']['IOType'] = 'io_stream'

config['LayerName']['qconv_1']['ReuseFactor'] = 9
config['LayerName']['qconv_1']['ParallelizationFactor'] = 1
config['LayerName']['qconv_1']['Trace'] = True

config['LayerName']['qconv_2']['ReuseFactor'] = 36
config['LayerName']['qconv_2']['ParallelizationFactor'] = 1
config['LayerName']['qconv_2']['Trace'] = True

config['LayerName']['qconv_3']['ReuseFactor'] = 36
config['LayerName']['qconv_3']['ParallelizationFactor'] = 1
config['LayerName']['qconv_3']['Trace'] = True

config['LayerName']['qdense_1']['ReuseFactor'] = 28
config['LayerName']['qdense_1']['Trace'] = True
config['LayerName']['qdense_2']['ReuseFactor'] = 28
config['LayerName']['qdense_2']['Trace'] = True

config['LayerName']['output']['ReuseFactor'] = 24
config['LayerName']['output']['Trace'] = True

# Convert model
hls_model = hls4ml.converters.convert_from_keras_model(
    model_qat,
    hls_config=config,
    output_dir=output_dir,
    backend='Vitis',
    part='xc7z020clg400-1', # Pynq-Z2
    board='pynq-z2',
    io_type='io_stream',
    clock_period="10ns",
    clock_uncertainty='20.0%',
    input_type="float", 
    output_type="float",
    input_data_tb=X_TEST_PATH,
    output_data_tb=Y_TEST_PATH,
)

/home/jeison/Nextcloud/PUCMM/ProyectoGrado/.venv/lib/python3.13/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'parallelization_factor' of layer 'qconv_1' (Conv2D):36 -> 1
  self.graph[name] = self.make_node(kind, name, layer, inputs, outputs)
/home/jeison/Nextcloud/PUCMM/ProyectoGrado/.venv/lib/python3.13/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'parallelization_factor' of layer 'qconv_2' (Conv2D):16 -> 1
  self.graph[name] = self.make_node(kind, name, layer, inputs, outputs)
/home/jeison/Nextcloud/PUCMM/ProyectoGrado/.venv/lib/python3.13/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'parallelization_factor' of layer 'qconv_3' (Conv2D):4 -> 1
  self.graph[name] = self.make_node(kind, name, layer, inputs, outputs)


## Profiling

In [8]:
# Compile and synthesize the model
hls_model.compile()

# Shows the median and quartiles of weight/activation distributions with box-and-whisker diagrams
# Grey box should cover the full range of the box-and-whisker plot

if PROFILE:
    wp, wph, ap, aph = numerical(
        model=model_qat,
        hls_model=hls_model,
        X=X_test,
        plot='boxplot'
    )

### Model

In [9]:
precision_info = inspect_hls4ml_precision(
    hls_model,
    report=True,
    save=True,
    model_name="hls_model",
)


hls4ml Layer Precision Report:

Layer: pressure_map (Input)
  pressure_map: ufixed<7,8,RND,WRAP,0>

Layer: qconv_1 (Conv2D)
  accum_t:  fixed<14,10,TRN,WRAP,0>
  weight: fixed<7,2,TRN,WRAP,0>
  bias: ufixed<2,32,TRN,WRAP,0>
  qconv_1: fixed<10,9,RND,WRAP,0>

Layer: qbn_conv_1 (BatchNormalization)
  scale: ufixed<3,-2,TRN,WRAP,0>
  bias: fixed<7,3,TRN,WRAP,0>
  qbn_conv_1: fixed<8,7,TRN,WRAP,0>

Layer: qconv_act_1 (Activation)
  qconv_act_1: ufixed<5,5,RND,WRAP,0>

Layer: qconv_2 (Conv2D)
  accum_t:  fixed<15,10,TRN,WRAP,0>
  weight: fixed<6,1,TRN,WRAP,0>
  bias: ufixed<2,32,TRN,WRAP,0>
  qconv_2: fixed<10,7,RND,WRAP,0>

Layer: qbn_conv_2 (BatchNormalization)
  scale: ufixed<5,0,TRN,WRAP,0>
  bias: fixed<7,3,TRN,WRAP,0>
  qbn_conv_2: fixed<10,7,TRN,WRAP,0>

Layer: qconv_act_2 (Activation)
  qconv_act_2: ufixed<6,4,RND,WRAP,0>

Layer: qconv_3 (Conv2D)
  accum_t:  fixed<16,9,TRN,WRAP,0>
  weight: fixed<6,1,TRN,WRAP,0>
  bias: ufixed<2,32,TRN,WRAP,0>
  qconv_3: fixed<9,6,RND,WRAP,0>

Laye

## Building

### Soft IP

In [10]:
# Build with various options
if BUILD_IP:
    report = hls_model.build(
            reset=RESET_BUILD,   # Reset project before build
            csim=False,          # Run C simulation
            synth=True,          # Run HLS synthesis
            cosim=False,         # Run co-simulation
            fifo_opt=False,      # FIFO optimization
            log_to_stdout=True,
        )


****** vitis-run v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [vitis-run 82-31] Launching vitis_hls: vitis_hls -nolog -run tcl -f /home/jeison/Nextcloud/PUCMM/ProyectoGrado/pipeline/output/export_ip/build_prj.tcl -work_dir /home/jeison/Nextcloud/PUCMM/ProyectoGrado/pipeline/output/export_ip

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2023.2 (64-bit)
  **** SW Build 4023990 on Oct 11 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source /home/jeison/AMD/Vitis_HLS/2023.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/home/jeison/AMD/Vitis_HLS/2023.2/bin/unwrapped/lnx64.o/viti

### Report

In [11]:
if BUILD_IP:
    # Latency (cycles)
    best_latency = int(report['CSynthesisReport']['BestLatency'])
    worst_latency = int(report['CSynthesisReport']['WorstLatency'])
    print(f"Latency: {best_latency} - {worst_latency} cycles")

    # Clock (ns)
    target_clock = float(report['CSynthesisReport']['TargetClockPeriod'])
    estimated_clock = float(report['CSynthesisReport']['EstimatedClockPeriod'])
    print(f"Target Clock = {target_clock} ns | Estimated Clock = {estimated_clock} ns")

    # Available resources
    avail_LUT = int(report['CSynthesisReport']['AvailableLUT'])
    avail_FF = int(report['CSynthesisReport']['AvailableFF'])
    avail_DSP = int(report['CSynthesisReport']['AvailableDSP'])
    avail_BRAM = int(report['CSynthesisReport']['AvailableBRAM_18K'])

    # Used resources (Vitis HLS)
    used_LUT = int(report['CSynthesisReport']['LUT'])
    used_FF = int(report['CSynthesisReport']['FF'])
    used_DSP = int(report['CSynthesisReport']['DSP'])
    used_BRAM = int(report['CSynthesisReport']['BRAM_18K'])

    # Calculate percentages
    used_LUT_pct = (used_LUT / avail_LUT) * 100
    used_FF_pct = (used_FF / avail_FF) * 100
    used_DSP_pct = (used_DSP / avail_DSP) * 100
    used_BRAM_pct = (used_BRAM / avail_BRAM) * 100

    # Vitis HLS Resources
    print(f"\n--- Vitis HLS Resource Utilization ---")
    print(f"LUT:  {used_LUT:>6} / {avail_LUT:>6} ({used_LUT_pct:>5.2f}%)")
    print(f"FF:   {used_FF:>6} / {avail_FF:>6} ({used_FF_pct:>5.2f}%)")
    print(f"DSP:  {used_DSP:>6} / {avail_DSP:>6} ({used_DSP_pct:>5.2f}%)")
    print(f"BRAM: {used_BRAM:>6} / {avail_BRAM:>6} ({used_BRAM_pct:>5.2f}%)")

Latency: 1569 - 1602 cycles
Target Clock = 10.0 ns | Estimated Clock = 9.4 ns

--- Vitis HLS Resource Utilization ---
LUT:   45593 /  53200 (85.70%)
FF:    26789 / 106400 (25.18%)
DSP:     219 /    220 (99.55%)
BRAM:    129 /    280 (46.07%)


In [12]:
if BUILD_IP:
    src_verilog = Path("output/export_ip/myproject_prj/solution1/impl/verilog")
    dst_rtl = Path("rtl")

    if not src_verilog.exists():
        raise FileNotFoundError(f"RTL source not found: {src_verilog.resolve()}")

    if dst_rtl.exists():
        shutil.rmtree(dst_rtl)

    shutil.copytree(src_verilog, dst_rtl)
    print(f"RTL successfully copied from {src_verilog.resolve()} to {dst_rtl.resolve()}")

RTL successfully copied from /home/jeison/Nextcloud/PUCMM/ProyectoGrado/pipeline/output/export_ip/myproject_prj/solution1/impl/verilog to /home/jeison/Nextcloud/PUCMM/ProyectoGrado/pipeline/rtl
